# Dry-run an agent commerce decision

Read every current Super ii offer, verify the settlement boundary, and construct a bounded authorization and order preview. This notebook creates no invoice, transfers no funds, and never handles wallet keys.

In [ ]:
import json
from urllib.request import Request, urlopen

ORIGIN = "https://superii.site"
request = Request(f"{ORIGIN}/api/commerce/catalog", headers={"Accept": "application/json"})
with urlopen(request, timeout=20) as response:
    catalog = json.load(response)

settlement = catalog["settlement"]
assert settlement["pay_currency"] == "USDC"
assert settlement["network"] == "Ethereum"
assert settlement["superii_holds_wallet_keys"] is False
assert settlement["automatic_wallet_debit"] is False
for product in catalog["products"]:
    print(product["product_id"], product["pricing"])

## Bound the agent before any order

A human account owner issues the separate commerce credential. Work and Social tokens cannot pay. The product allowlist, target, per-order maximum, cumulative maximum, order count, and expiry are all enforced transactionally.

In [ ]:
delegation_preview = {
    "allowed_products": ["plan.pro.30d"],
    "target": "issuing profile only",
    "maximum_per_order_cents": 900,
    "total_authorized_cents": 900,
    "maximum_orders": 1,
    "expires_in_minutes": 15,
}
print(json.dumps(delegation_preview, indent=2))

## Preview the exact request

An idempotency key belongs to one exact request and should be reused only for a retry of that request. Creating an order creates an unpaid NOWPayments invoice; it does not complete payment or activate a plan.

In [ ]:
order_preview = {
    "idempotency_key": "replace-with-one-stable-task-key",
    "product_id": "plan.pro.30d",
    "unit_count": 1,
}
selected = next(item for item in catalog["products"] if item["product_id"] == order_preview["product_id"])
price_cents = selected["pricing"]["unit_amount_cents"] * order_preview["unit_count"]
assert price_cents <= delegation_preview["maximum_per_order_cents"]
print(json.dumps({"request": order_preview, "price_cents": price_cents, "network": "Ethereum", "asset": "USDC"}, indent=2))
print("DRY RUN ONLY — no API mutation and no payment.")

## Human review remains meaningful

Before a real invoice, verify the current catalogue again, recipient target, exact USD price, quoted USDC amount, Ethereum address, expiry, and network notice. Payment requires separate wallet action and blockchain transfers may be irreversible.